
# Reproducible BoQ-to-LCA demonstration

This notebook reproduces the **computational structure** of the proposed workflow using only synthetic, redistribution-safe inputs.

It does **not** contain the original project BoQ, BCCA database content, ecoinvent data, or any confidential/project-specific files.

The exact numerical results in the paper require authorised access to the original project documentation and the same licensed/reference datasets used by the authors.


## Methodological references

This public demonstration implements methods described in:

1. S. Gachkar et al., *Text-based algorithms for automating life cycle inventory analysis in building sector life cycle assessment studies*, Journal of Cleaner Production 486 (2025) 144448. DOI: https://doi.org/10.1016/j.jclepro.2024.144448
2. D. Gachkar et al., *Automating data integration for construction Life Cycle Assessment using fuzzy matching and supervised learning*, Automation in Construction 178 (2025) 106381. DOI: https://doi.org/10.1016/j.autcon.2025.106381

The notebook uses only synthetic demonstration inputs and does not redistribute the project BoQ, BCCA data, or ecoinvent data.


In [ ]:

from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.workflow import load_stopwords, run_from_pdf
from src.sensitivity import run_matching_sensitivity

DATA = ROOT / "data" / "example"


## Load synthetic example inputs

In [ ]:

reference_db = pd.read_csv(DATA / "reference_database.csv")
impact_factors = pd.read_csv(DATA / "impact_factors.csv")
stop_words = load_stopwords(DATA / "spanish_stopwords.txt")

reference_db.head()


## Run PDF extraction, task matching, material aggregation, and impact calculation

In [ ]:

run = run_from_pdf(
    boq_pdf=DATA / "demo_boq.pdf",
    reference_database=reference_db,
    impact_factors=impact_factors,
    stop_words=stop_words,
)

print("Extracted tasks")
display(run["tasks"])

print("Material inventory")
display(run["inventory"][[
    "Material Code", "Material Title", "Material Unit", "Total Material"
]])

print("Impact results")
display(run["results"])

print(f"Synthetic total A1-A3 impact: {run['total_impact']:.4f}")
print(f"Runtime: {run['runtime_seconds']:.3f} s")



## Optional matching-error sensitivity demonstration

This is the same *type* of Monte Carlo stress test described in the revised manuscript. The synthetic results below are for demonstration only and are not the paper's case-study values.


In [ ]:

sensitivity = run_matching_sensitivity(
    run["results"],
    error_rates=(0.05, 0.10, 0.20),
    n_simulations=1000,
    seed=42,
)
sensitivity



## Data-access note

The repository separates **code reproducibility** from **third-party data redistribution**:

- the extraction and processing code is public;
- the example PDF and tables are synthetic;
- the original project BoQ is intentionally excluded;
- third-party construction/environmental database content is intentionally excluded;
- exact reproduction of the paper's numerical case-study result requires authorised access to those original inputs.
